# Классификация кристаллических структур урановых соединений

## Расширенный датасет:
- **NaCl** (7x7x7) - структура хлорида натрия
- **UN2** - динитрид урана
- **U2N3 (alpha)** - кубическая фаза (Ia-3)
- **U2N3 (beta)** - тригональная фаза (P-3m1)
- **UC** - монокарбид урана (NaCl-type)
- **UC2 (alpha)** - тетрагональная фаза (I4/mmm)
- **UC2 (beta)** - кубическая фаза (Fm-3m)
- **U2C3** - сесквикарбид урана (I-43d)


In [ ]:
from google.colab import files
import zipfile
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Загрузка архива
uploaded = files.upload()
archive_name = list(uploaded.keys())[0]

# Распаковка архива
with zipfile.ZipFile(archive_name, 'r') as zip_ref:
    zip_ref.extractall('data')

data_dir = 'data'

def load_lattice_data(lattice_dir, label):
    """
    Загрузка данных для одного типа решётки
    """
    lattice_data = []
    lattice_labels = []
    for lattice_num in os.listdir(lattice_dir):
        lattice_path = os.path.join(lattice_dir, lattice_num)
        if os.path.isdir(lattice_path):
            kde_data = []
            for filename in os.listdir(lattice_path):
                filepath = os.path.join(lattice_path, filename)
                kde_values = pd.read_csv(filepath, header=0).iloc[:, 0].values[1:]
                kde_data.append(kde_values.astype(float))
            kde_matrix = np.column_stack(kde_data)
            lattice_data.append(kde_matrix)
            lattice_labels.append(label)
    return lattice_data, lattice_labels

# Загрузка данных для каждого типа решётки
print("Загрузка данных...")

# Оригинальные структуры
lattice_data_NaCl, labels_NaCl = load_lattice_data(
    os.path.join(data_dir, '7x7x7'), 'NaCl')
lattice_data_UN2, labels_UN2 = load_lattice_data(
    os.path.join(data_dir, 'UN2'), 'UN2')
lattice_data_U2N3, labels_U2N3 = load_lattice_data(
    os.path.join(data_dir, 'U2N3'), 'U2N3')

# Новые структуры нитридов
lattice_data_U2N3_alpha, labels_U2N3_alpha = load_lattice_data(
    os.path.join(data_dir, 'U2N3_alpha'), 'U2N3_alpha')
lattice_data_U2N3_beta, labels_U2N3_beta = load_lattice_data(
    os.path.join(data_dir, 'U2N3_beta'), 'U2N3_beta')

# Карбиды урана
lattice_data_UC, labels_UC = load_lattice_data(
    os.path.join(data_dir, 'UC'), 'UC')
lattice_data_UC2_alpha, labels_UC2_alpha = load_lattice_data(
    os.path.join(data_dir, 'UC2_alpha'), 'UC2_alpha')
lattice_data_UC2_beta, labels_UC2_beta = load_lattice_data(
    os.path.join(data_dir, 'UC2_beta'), 'UC2_beta')
lattice_data_U2C3, labels_U2C3 = load_lattice_data(
    os.path.join(data_dir, 'U2C3'), 'U2C3')

# Объединение всех данных
X = (lattice_data_NaCl + lattice_data_UN2 + lattice_data_U2N3 +
     lattice_data_U2N3_alpha + lattice_data_U2N3_beta +
     lattice_data_UC + lattice_data_UC2_alpha + lattice_data_UC2_beta +
     lattice_data_U2C3)

y = np.hstack([
    np.array(labels_NaCl),
    np.array(labels_UN2),
    np.array(labels_U2N3),
    np.array(labels_U2N3_alpha),
    np.array(labels_U2N3_beta),
    np.array(labels_UC),
    np.array(labels_UC2_alpha),
    np.array(labels_UC2_beta),
    np.array(labels_U2C3)
])

print(f"\nВсего загружено образцов: {len(X)}")
print(f"Распределение по классам:")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count}")

def pad_matrix(matrix, max_columns):
    """
    Дополнение матрицы нулями до нужного размера
    """
    padded_matrix = np.pad(
        matrix, ((0, 0), (0, max_columns - matrix.shape[1])), 
        mode='constant'
    )
    return padded_matrix

# Находим максимальное количество ионов в одной решётке
max_ions = max(matrix.shape[1] for matrix in X)
print(f"\nМаксимальное число ионов в решётке: {max_ions}")

# Дополняем все матрицы до одинакового размера
X_padded = [pad_matrix(matrix, max_ions) for matrix in X]

# Уплощаем матрицы в вектора
def matrix_to_vector(matrix):
    return matrix.flatten()

X_flattened = [matrix_to_vector(matrix) for matrix in X_padded]
X_flattened = np.array(X_flattened)

print(f"Размер признакового пространства: {X_flattened.shape}")

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_flattened, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nРазмер обучающей выборки: {X_train.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")


In [ ]:
# Создание и обучение базовой модели Random Forest
print("\n=== Обучение модели Random Forest ===")
model_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

# Предсказание на тестовой выборке
y_pred_rf = model_rf.predict(X_test)

# Оценка точности
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy (Random Forest): {accuracy_rf:.4f}")

# Детальный отчёт по классификации
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# Матрица ошибок
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Подбор гиперпараметров с помощью Grid Search
print("\n=== Подбор гиперпараметров ===")
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nЛучшие параметры: {grid_search.best_params_}")
print(f"Лучший score на кросс-валидации: {grid_search.best_score_:.4f}")

# Обучение модели с лучшими параметрами
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
print(f"\nBest Model Accuracy: {accuracy_best:.4f}")

# Сохранение лучшей модели
joblib.dump(best_model, 'best_uranium_classifier.pkl')
print("\nМодель сохранена как 'best_uranium_classifier.pkl'")

# Важность признаков (топ-20)
feature_importance = best_model.feature_importances_
top_features_idx = np.argsort(feature_importance)[-20:]

plt.figure(figsize=(10, 8))
plt.barh(range(20), feature_importance[top_features_idx])
plt.xlabel('Feature Importance')
plt.ylabel('Feature Index')
plt.title('Top 20 Most Important Features')
plt.yticks(range(20), top_features_idx)
plt.tight_layout()
plt.show()


In [ ]:
# Сравнение различных классификаторов
print("\n=== Сравнение классификаторов ===")

models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

results = {}
for name, model in models.items():
    print(f"\nОбучение {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name} Accuracy: {accuracy:.4f}")

# Визуализация сравнения
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values())
plt.xlabel('Classifier')
plt.ylabel('Accuracy')
plt.title('Comparison of Different Classifiers')
plt.ylim([0, 1])
for i, (name, acc) in enumerate(results.items()):
    plt.text(i, acc + 0.01, f'{acc:.4f}', ha='center')
plt.tight_layout()
plt.show()


## Инструкция по подготовке данных

### Структура директорий для новых данных:

```
data/
├── 7x7x7/              # NaCl структура (уже есть)
├── UN2/                # Динитрид урана (уже есть)
├── U2N3/               # Оригинальный U2N3 (уже есть)
├── U2N3_alpha/         # Кубическая фаза U2N3 (NEW)
├── U2N3_beta/          # Тригональная фаза U2N3 (NEW)
├── UC/                 # Монокарбид урана (NEW)
├── UC2_alpha/          # Тетрагональная фаза UC2 (NEW)
├── UC2_beta/           # Кубическая фаза UC2 (NEW)
└── U2C3/               # Сесквикарбид урана (NEW)
```

### Для каждой новой структуры:

1. Использовать соответствующий CIF файл из outputs/
2. Сгенерировать KDE данные используя тот же метод, что и для оригинальных структур
3. Создать поддиректории для каждой решётки (lattice_0, lattice_1, ...)
4. Поместить CSV файлы с KDE данными в соответствующие папки

### Файлы CIF созданы:
- `alpha-U2N3_mp-22387.cif` - для U2N3_alpha
- `beta-U2N3_mp-973.cif` - для U2N3_beta
- `UC_mp-2489.cif` - для UC
- `alpha-UC2_mp-2486.cif` - для UC2_alpha
- `beta-UC2_mp-1008642.cif` - для UC2_beta
- `U2C3_mp-2625.cif` - для U2C3
